# Sistema de Recuperacion de Informacion Multimodal con RAG

**Asignatura:** Recuperacion de Informacion
**Corpus:** Amazon Shopping Queries Dataset (ESCI) + SQID (Shopping Queries Image Dataset)

- Nombre(s):
- Fecha:

---

Este notebook orquesta el pipeline completo llamando a las funciones definidas en `src/`.
No se reimplementa logica aqui: cada fase importa y ejecuta los modulos correspondientes.

**Estructura:** primero se construye todo el sistema (corpus, embeddings, indice FAISS,
componentes de RAG y de las funcionalidades de excelencia). Las pruebas, demostraciones y
resultados con multiples queries de ejemplo estan al final, en la seccion **"Pruebas y
resultados"**.


In [ ]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd()))

import pandas as pd
from IPython.display import display, Image as IPImage

from src import config


---
## Fase A: Preparacion del corpus

Se unen dos fuentes:
- **ESCI** (`tasksource/esci`, subconjunto test/us/small_version=1): texto de queries, texto de
  productos y etiquetas de relevancia (`esci_label`) que usamos como *qrels*.
- **SQID** (`crossingminds/shopping-queries-image-dataset`): URLs de imagen de cada producto.

El join se hace por `product_id` (ASIN). Luego se muestrea un subconjunto de queries
(manteniendo TODOS sus productos asociados, para preservar qrels validos) y se descargan
las imagenes localmente.


In [ ]:
from src.data_loading import build_joined_dataset

joined = build_joined_dataset()
print("Filas unidas (query, producto):", joined.shape)
print("Productos unicos:", joined.product_id.nunique())
print("Queries unicas:", joined.query_id.nunique())
print("Cobertura de imagen:", f"{joined.image_url.notna().mean():.2%}")
joined.head(3)


In [ ]:
from src.corpus_builder import build_corpus

# force_rebuild=False reutiliza el corpus ya construido en data/processed/ si existe
products, qrels_sample = build_corpus(force_rebuild=False)

print("Productos en el corpus (con imagen descargada):", products.shape)
print("Pares (query, producto) para evaluacion:", qrels_sample.shape)
print("Queries de evaluacion:", qrels_sample.query_id.nunique())
products[["product_id", "product_title", "document", "image_path"]].head(3)


---
## Fase B: Representacion mediante Embeddings (CLIP)

Se usa un modelo CLIP (`openai/clip-vit-base-patch32` via `transformers`) para proyectar
texto e imagen al mismo espacio vectorial. Cada producto se representa con un unico vector
multimodal (promedio ponderado de su embedding de texto y de imagen, renormalizado).


In [ ]:
from src.embeddings import ClipEmbedder
import numpy as np
from tqdm.auto import tqdm

embedder = ClipEmbedder()

product_embeddings = np.stack([
    embedder.encode_multimodal_document(row.document, row.image_path)
    for row in tqdm(products.itertuples(index=False), total=len(products), desc="Embeddings CLIP")
])

print("Shape embeddings:", product_embeddings.shape)
print("Norma del primer vector (debe ser ~1.0):", np.linalg.norm(product_embeddings[0]))


---
## Fase C: Base de datos vectorial (FAISS)

Se indexan los embeddings en un `IndexFlatIP` (producto interno sobre vectores normalizados
= similitud coseno). El indice y sus metadatos se persisten en `data/faiss_index/`.


In [ ]:
from src.vector_store import FaissVectorStore

store = FaissVectorStore(dim=product_embeddings.shape[1])
store.build(product_embeddings, products)
store.save(config.INDEX_DIR)
print("Documentos indexados:", store.index.ntotal)


---
## Fase D: Sistema RAG (retrieve -> context -> generate)

`src/rag.py` implementa el pipeline completo: recuperacion (con **Query Expansion** opcional
via LLM), **re-ranking** con CrossEncoder, construccion de contexto y generacion de respuesta
con Gemini. La funcion devuelve tambien las evidencias usadas (texto, imagen, score) para
cumplir el requisito de transparencia/trazabilidad.

En esta fase solo se instancian los componentes y se define la funcion de despliegue de
resultados; las ejecuciones de ejemplo estan en la seccion final de pruebas.

> Requiere `GEMINI_API_KEY` configurada en un archivo `.env` en la raiz del proyecto.


In [ ]:
from src.reranker import Reranker

reranker = Reranker()


In [ ]:
def display_rag_result(result, max_chars=300):
    print("=" * 90)
    print("CONSULTA:", result["query"])
    if result["effective_query"] != result["query"]:
        print("CONSULTA EFECTIVA (memoria):", result["effective_query"])
    print("=" * 90)
    print("\nRESPUESTA GENERADA:\n")
    print(result["answer"])
    print("\n" + "-" * 90)
    print(f"EVIDENCIAS UTILIZADAS ({len(result['evidences'])}):")
    print("-" * 90)
    for i, ev in enumerate(result["evidences"], start=1):
        score = ev.get("rerank_score", ev.get("score"))
        print(f"\n[Evidencia {i}] score: {score:.4f} | {ev['product_title']}")
        if ev.get("image_path"):
            display(IPImage(filename=ev["image_path"], width=120))
        texto = ev["document"]
        print(texto[:max_chars] + ("..." if len(texto) > max_chars else ""))
    print("\n" + "=" * 90)


---
## Fase E: Interfaz conversacional (Streamlit)

La interfaz de chat vive en `app/streamlit_app.py` y reutiliza exactamente estos mismos
modulos (`ClipEmbedder`, `FaissVectorStore`, `Reranker`, `rag_pipeline`, `ConversationMemory`,
`FeedbackStore`). Presenta la respuesta, las imagenes/evidencias recuperadas, el score de
cada una, botones de "Me gusta"/"No me gusta" (relevance feedback) y mantiene memoria de la
conversacion.

Para ejecutarla:
```bash
streamlit run app/streamlit_app.py
```


---
## Fase F: Evaluacion experimental

Se evalua el sistema de recuperacion (FAISS + CLIP, con y sin re-ranking) contra los qrels
reales derivados de `esci_label` (Exact/Substitute/Complement/Irrelevant) para las queries
muestreadas. Metricas: **Precision@k**, **Recall@k**, **NDCG@k** (k = 3, 5, 10).

Aqui solo se define el conjunto de queries de evaluacion y las funciones de busqueda; la
ejecucion de la evaluacion esta en la seccion final de pruebas.


In [ ]:
from src.evaluation import evaluate_system, summarize

eval_queries = qrels_sample[["query_id", "query"]].drop_duplicates().reset_index(drop=True)
print("Queries de evaluacion disponibles:", len(eval_queries))


def search_fn_base(query_text):
    q_emb = embedder.encode_query(query_text)
    hits = store.search(q_emb, top_k=10)
    return [h["product_id"] for h in hits]


def search_fn_reranked(query_text):
    q_emb = embedder.encode_query(query_text)
    candidates = store.search(q_emb, top_k=config.TOP_K_RETRIEVE)
    reranked = reranker.rerank(query_text, candidates, top_k=10)
    return [h["product_id"] for h in reranked]


---
## Funcionalidades de excelencia (componentes)

- **Re-ranking (+15)**: ya instanciado arriba (`reranker`).
- **Query Expansion (+15)**: `src/query_expansion.expand_query` / `multi_query_retrieve`, sin
  estado, se usa directo en la seccion de pruebas.
- **Relevance Feedback (+15)**: `FeedbackStore`, mantiene estado por sesion.
- **Memoria conversacional (+15)**: `ConversationMemory`, mantiene el historial de turnos.

Se instancian aqui los componentes con estado; se ejercitan en la seccion final.


In [ ]:
from src.feedback import FeedbackStore
from src.memory import ConversationMemory

feedback = FeedbackStore()
memory = ConversationMemory()


---
---

# Pruebas y resultados

A partir de aca se ejercita todo lo construido arriba con **multiples queries de ejemplo**.
Se usa una query conocida ("wireless bluetooth headphones", ya validada manualmente) mas
varias queries reales tomadas al azar del propio conjunto de evaluacion (`qrels_sample`), de
forma que las pruebas queden ancladas a productos con relevancia juzgada realmente en el
corpus muestreado.


In [ ]:
KNOWN_GOOD_QUERY = "wireless bluetooth headphones"

sampled_queries = (
    qrels_sample["query"]
    .drop_duplicates()
    .sample(n=5, random_state=config.RANDOM_SEED)
    .tolist()
)

TEST_QUERIES = [KNOWN_GOOD_QUERY] + [q for q in sampled_queries if q != KNOWN_GOOD_QUERY]

print(f"Queries de prueba ({len(TEST_QUERIES)}):")
for q in TEST_QUERIES:
    print(" -", q)


## Prueba 1: Recuperacion FAISS pura (sin RAG)

Similitud coseno directa entre el embedding CLIP de la query y el indice, sin re-ranking ni
generacion.


In [ ]:
for q in TEST_QUERIES:
    print("=" * 90)
    print("QUERY:", q)
    hits = store.search(embedder.encode_query(q), top_k=5)
    for i, h in enumerate(hits, start=1):
        print(f"  [{i}] score={h['score']:.4f} | {h['product_title']}")
print("=" * 90)


## Prueba 2: Pipeline RAG completo (retrieve + rerank + generar)

Para cada query: recuperacion con query expansion, re-ranking con CrossEncoder, generacion
con Gemini y despliegue de evidencias (texto + imagen + score).


In [ ]:
from src.rag import rag_pipeline

rag_results = []
for q in TEST_QUERIES:
    result = rag_pipeline(
        q,
        embedder=embedder,
        store=store,
        reranker=reranker,
        use_query_expansion=True,
    )
    rag_results.append(result)
    display_rag_result(result)


## Prueba 3: Evaluacion cuantitativa (Precision@k, Recall@k, NDCG@k)

Evaluacion sobre **todas** las queries del conjunto muestreado (no solo `TEST_QUERIES`),
comparando el ranking base (solo FAISS/CLIP) contra el ranking con re-ranking.


In [ ]:
results_base = evaluate_system(search_fn_base, eval_queries, qrels_sample)
results_reranked = evaluate_system(search_fn_reranked, eval_queries, qrels_sample)

print("=== Solo FAISS (CLIP) ===")
print(summarize(results_base))
print("\n=== FAISS + Re-ranking (CrossEncoder) ===")
print(summarize(results_reranked))


Comparar `results_base` vs `results_reranked` permite cuantificar el aporte del
re-ranking (funcionalidad de excelencia) sobre la calidad del ranking inicial. Para el
informe tecnico, incluir esta tabla y un analisis de en que tipo de consultas mejora o
empeora el re-ranking.


## Prueba 4: Query Expansion

Reformulaciones generadas por el LLM para algunas de las queries de prueba.


In [ ]:
from src.query_expansion import expand_query

for q in TEST_QUERIES[:3]:
    print("QUERY:", q)
    print("  Expansiones:", expand_query(q))
    print()


## Prueba 5: Relevance Feedback (Rocchio)

Simulacion de un usuario marcando "me gusta"/"no me gusta" sobre los resultados de
`KNOWN_GOOD_QUERY`, y como afecta la siguiente busqueda de la misma sesion via el ajuste de
Rocchio sobre el vector de consulta.


In [ ]:
session_id = "demo-session"

first_pass = store.search(embedder.encode_query(KNOWN_GOOD_QUERY), top_k=5)
print("Antes de feedback:")
for r in first_pass:
    print(" -", r["product_title"])

# El usuario marca el primer resultado como 'no me gusta'
feedback.record(session_id, first_pass[0]["product_id"], liked=False)
# y el segundo como 'me gusta'
feedback.record(session_id, first_pass[1]["product_id"], liked=True)

adjusted_query = feedback.apply_rocchio(session_id, embedder.encode_query(KNOWN_GOOD_QUERY), store)
second_pass = store.search(adjusted_query, top_k=5)

print("\nDespues de feedback:")
for r in second_pass:
    print(" -", r["product_title"])


## Prueba 6: Memoria conversacional

El sistema reformula preguntas de seguimiento en consultas autocontenidas usando el
historial de la conversacion (turno 1 con `KNOWN_GOOD_QUERY`, turno 2 una pregunta de
seguimiento que depende del contexto).


In [ ]:
r1 = rag_pipeline(KNOWN_GOOD_QUERY, embedder=embedder, store=store,
                   reranker=reranker, memory=memory, use_query_expansion=False)
display_rag_result(r1)

r2 = rag_pipeline("y en color negro?", embedder=embedder, store=store,
                   reranker=reranker, memory=memory, use_query_expansion=False)
display_rag_result(r2)


---
## Notas finales

- Todo el codigo reutilizable vive en `src/` (y `app/streamlit_app.py` para la interfaz).
  Este notebook solo orquesta las llamadas, en linea con la consigna del proyecto.
- El tamano de muestra (`config.N_QUERIES_SAMPLE`, `config.MAX_PRODUCTS_SAMPLE`) esta
  ajustado para poder correr CLIP en CPU en tiempos razonables. Ver `README.md` para
  instrucciones de instalacion y ejecucion completas.
